# 01 · Preparación de datos (ESCI en español)

**Laboratorio 4 · Asistente de compras por voz** (MCD8016 Deep Learning, UTEC)

Este notebook produce los datos que piden la sección 2 del enunciado y que usan los demás notebooks:

| Salida | Carpeta | ¿Lo ve el agente? | Contenido |
|---|---|---|---|
| `catalog.parquet` | `processed/` | **Sí** | ≥ 5 000 productos: id, título, descripción, bullets, marca y color |
| `queries_dev.parquet` | `processed/` | Sí (desarrollo) | Consultas de `split = train` para ajustar el método |
| `queries_test.parquet` | `processed/` | Sí (evaluación) | ≥ 200 consultas de `split = test`, **solo** para reportar |
| `qrels_{dev,test}.parquet` | `eval/` | **No** | Etiquetas ESCI y ganancias. Las lee solo el módulo de evaluación |
| `manifest.json` | raíz de datos | — | Configuración, conteos y huellas para verificar la reproducibilidad |

**Reglas del enunciado que se cumplen aquí:**
1. Versión reducida (`small_version = 1`), filtrada al español (`product_locale = es`).
2. Las consultas se separan por `query_id` respetando el `split` original de ESCI.
3. Todos los productos anotados de las consultas elegidas están en el catálogo.
4. Las etiquetas ESCI se guardan aparte para que el agente no pueda consultarlas.
5. Todo depende de una semilla fija (`SEED = 42`), de modo que la Mac y Colab generan los mismos datos.

> La lógica vive en `src/asistente_compras/data.py`. El notebook la ejecuta y muestra un EDA breve para justificar las decisiones.

## 0. Entorno

In [ ]:
# ── Configuración del entorno (idéntica en todos los notebooks) ──────────────────────────
# Detectar si corre en Colab o en la Mac y dejar importable el paquete `asistente_compras`.
import os
import sys
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    # 1) Montar Drive: guarda el código y los datos entre sesiones (Colab borra /content al cerrar).
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/lab4/lab4_solution")
    os.environ["LAB4_DATA_DIR"] = "/content/drive/MyDrive/lab4/data"
    # 2) Instalar el paquete y sus dependencias. EXTRAS cambia según el notebook
    #    ("" = núcleo; "rag", "llm", "voz" en los siguientes).
    EXTRAS = ""
    target = f"{PROJECT_DIR}[{EXTRAS}]" if EXTRAS else str(PROJECT_DIR)
    get_ipython().run_line_magic("pip", f"install -q -e {target}")
else:
    # En la Mac, usar como kernel el entorno uv del proyecto (ver README).
    PROJECT_DIR = Path.cwd().resolve().parent

# Una instalación editable recién hecha no se ve hasta reiniciar el kernel;
# agregar src/ al path para evitar ese reinicio.
sys.path.insert(0, str(PROJECT_DIR / "src"))
print(f"Colab: {EN_COLAB} | proyecto: {PROJECT_DIR}")

In [ ]:
from dataclasses import asdict

import pandas as pd

from asistente_compras.config import DataConfig, get_paths
from asistente_compras.data import download_esci, load_examples_es, load_products_es

pd.set_option("display.max_colwidth", 80)

cfg = DataConfig()     # usar los parámetros por defecto (ver config.py)
paths = get_paths().ensure()
print("Datos en:", paths.root)
asdict(cfg)

## 1. Descarga

Los parquet de ESCI se distribuyen con Git LFS, así que se bajan directamente desde `media.githubusercontent.com`:
- `examples` ≈ 51 MB: pares consulta–producto con su etiqueta.
- `products` ≈ 1,1 GB: textos de los productos, en todos los idiomas.

Si los archivos ya existen, no se vuelven a descargar.

In [ ]:
raw_files = download_esci(paths)
{k: f"{v.stat().st_size / 1e6:,.0f} MB" for k, v in raw_files.items()}

## 2. EDA del subconjunto español (versión reducida)

Se filtra **al leer** el parquet (`filters=` de pyarrow). Cargar los 1,1 GB de productos de los tres idiomas usaría varios GB de RAM, y el Colab gratuito tiene unos 12 GB.

In [ ]:
examples = load_examples_es(paths, cfg.locale)
print(f"Pares consulta–producto: {len(examples):,}  |  consultas únicas: {examples.query_id.nunique():,}")

# Contar consultas y juicios por split (deben coincidir con la Tabla 1 del README de ESCI: 5 632 / 2 417 consultas)
examples.groupby("split").agg(consultas=("query_id", "nunique"), juicios=("product_id", "size"))

In [ ]:
# Ver la distribución de etiquetas ESCI: cuánto pesan Exact frente a Substitute/Complement/Irrelevant.
dist = examples.esci_label.value_counts(normalize=True).rename("proporción").to_frame()
dist["ganancia nDCG"] = dist.index.map(cfg.gains)
dist

In [ ]:
# Medir la profundidad: cuántos candidatos anotados tiene cada consulta.
# nDCG@10 se calcula re-ordenando estos candidatos, así que con menos de 10 el @10 cubre toda la lista.
depth = examples.groupby("query_id").size()
print(depth.describe().round(1).to_string())
print(f"\nConsultas con < 10 candidatos: {(depth < 10).mean():.1%}")
sin_relevantes = examples.groupby("query_id").esci_label.apply(lambda s: (s == "I").all())
print(f"Consultas cuyos candidatos son todos Irrelevant (se excluyen, nDCG indefinido): {sin_relevantes.sum()}")

In [ ]:
# Mostrar ejemplos de consultas reales: el tipo de lenguaje que tendrá que entender el agente.
examples.drop_duplicates("query_id").sample(10, random_state=cfg.seed)[["query_id", "query", "split"]]

In [ ]:
products = load_products_es(paths, cfg.locale)
print(f"Productos ES: {len(products):,}")

# Contar los campos vacíos por columna. Importa para la regla de que un atributo no verificable se trata como
# "desconocido": si la marca o el color faltan en la ficha, el agente no debe inventarlos.
vacios = products.drop(columns="product_id").apply(lambda s: s.isna() | (s.astype(str).str.strip() == ""))
vacios.mean().rename("% vacío").map("{:.1%}".format).to_frame()

## 3. Construcción del dataset

`prepare()` ejecuta en orden:
1. **Selección de consultas** con semilla: `n_test_queries` del split *test* y `n_dev_queries` del split *train*. Se excluyen solo las consultas sin ningún candidato relevante, para no sesgar la muestra.
2. **Catálogo** = unión de todos los productos anotados de esas consultas. Si no llega a 5 000, se completa con *distractores* (productos ES de otras consultas). Se limpia el HTML de descripciones y bullets.
3. **Chequeos de integridad**: anotados ⊂ catálogo, dev ∩ test = ∅ por `query_id` **y** por texto de consulta, y `product_id` único. Si alguno falla, se detiene.
4. **Escritura** de `processed/`, `eval/` y `manifest.json`.

In [ ]:
import json

from asistente_compras.data import prepare

del products  # liberar RAM antes de reconstruir (prepare vuelve a leer, ya filtrado)
manifest = prepare(cfg, paths)
print(json.dumps(manifest["counts"], indent=2))

## 4. Inspección de las salidas

In [ ]:
catalog = pd.read_parquet(paths.processed / "catalog.parquet")
q_test = pd.read_parquet(paths.processed / "queries_test.parquet")
q_dev = pd.read_parquet(paths.processed / "queries_dev.parquet")
print(f"Catálogo: {len(catalog):,} productos ({catalog.is_distractor.sum():,} distractores)")
print(f"Consultas: {len(q_dev)} dev | {len(q_test)} test")
catalog.head(3)

In [ ]:
# Medir la longitud del texto que se indexará (título + bullets + descripción).
# Usarla para decidir más adelante qué campos entran en el índice denso (límite de tokens de e5 ≈ 512).
texto = catalog.title + " " + catalog.bullet_points + " " + catalog.description
print(texto.str.split().str.len().describe(percentiles=[0.5, 0.9, 0.99]).round(0).to_string())

In [ ]:
# Comparar la distribución de etiquetas en test con el total ES: confirmar que la muestra es representativa.
qrels_test = pd.read_parquet(paths.eval / "qrels_test.parquet")
pd.DataFrame({
    "test elegido": qrels_test.esci_label.value_counts(normalize=True),
    "ES completo": examples.esci_label.value_counts(normalize=True),
}).round(3)

## 5. Reproducibilidad

`manifest.json` guarda una huella del **contenido** de cada tabla, que no depende de la versión de pyarrow. Al correr este notebook en Colab, las huellas deben coincidir con las generadas en la Mac. Si coinciden, ambos entornos evalúan exactamente sobre los mismos datos.

In [ ]:
manifest["fingerprints"]

### Resumen

- Se cumplen los mínimos del enunciado: catálogo ≥ 5 000 productos y ≥ 200 consultas de evaluación con todos sus anotados en el catálogo.
- Las etiquetas quedan en `eval/`, fuera del alcance del agente.
- **Siguiente:** `02_tienda_simulada.ipynb` (precios, stock, carritos y pedidos con semilla y restauración).